# PSU efficiency analysis

Perform all the PSU analysis presented in the IMC'25 paper (Section 9)

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

import plotly.graph_objects as go

## Initialization

In [2]:

# Select the different output format settings

PaperPlot = True
# PaperPlot = False
if PaperPlot:
    output_format = 'IMC'
else:
    output_format = 'online'

if output_format == 'online':
    font_size_px = 14
    linewidth_px = 512
    landscapewidth_px = 654
    plot_path = None
    
    plot_path = Path('plots')

if output_format == 'IMC':
    font_size_pt = 7
    offset = 5 # to compensate for the rounding of unit conversions
    linewidth_pt = 241 - offset  
    landscapewidth_pt = 506 - offset
    
    # 1pt = 1.333px
    font_size_px = int(font_size_pt*1.333)+1
    linewidth_px = int(linewidth_pt*1.333)+1
    landscapewidth_px = int(landscapewidth_pt*1.333)+1

    out_path = Path('output/2025_IMC/figures')

# Create the output directory if don't exist
Path(out_path).mkdir(parents=True, exist_ok=True)

# Input data
input_path = Path('input')

# Default plot layout
default_layout = {
    "title":None,
    "width":linewidth_px,
    "height":200,
    "font":{"size":font_size_px},
    "yaxis":{'title':{'font':{'size':font_size_px}}},
    "xaxis":{'title':{'font':{'size':font_size_px},
                      'text':'Time [ s ]'}}
}


## Efficient curve and 80 Plus standards

In [3]:
# ===============
# Import PFE data and derive the numerical models 
# for the different standard by adding a constant 
# to the PFE efficiency curve
# ===============

pfe_file = Path(input_path,'pfe-efficiency-curve.csv')
df_pfe = pd.read_csv(pfe_file)
df_pfe['load_%'] = (df_pfe['load_W'] / 600)*100

# .. Model is linear interpolation between the colected points
def eff_interp(x):
    return np.interp(x,df_pfe['load_%']/100,df_pfe['efficiency_%']/100)

marker_opacity = 1
marker_size = 9
marker_border_size = 1
marker_line = dict(
    width=marker_border_size,
    color='black'
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x = [10,20,50,100],
        y = [90,94,96,91],
        mode='markers',
        name='Titanium',
        marker=dict(
            symbol=19, 
            line=marker_line,
            size=marker_size, 
            color='rgb(121, 121, 130)', 
            ),
    )
)
fig.add_trace(
    go.Scatter(
        x = [20,50,100],
        y = [90,94,91],
        mode='markers',
        name='Platinum',
        marker=dict(symbol=0, 
            line=marker_line,size=marker_size, color='rgb(209, 208, 206)', ),
    )
)
fig.add_trace(
    go.Scatter(
        x = [20,50,100],
        y = [88,92,88],
        mode='markers',
        name='Gold',
        marker=dict(symbol=2,
            line=marker_line, size=marker_size, color='rgb(255, 215, 0)', opacity=marker_opacity),
    )
)
fig.add_trace(
    go.Scatter(
        x = [20,50,100],
        y = [85,89,85],
        mode='markers',
        name='Silver',
        marker=dict(symbol=4,
            line=marker_line, size=marker_size, color='rgb(192, 192, 192)', opacity=marker_opacity),
    )
)
fig.add_trace(
    go.Scatter(
        x = [20,50,100],
        y = [81,85,81],
        mode='markers',
        name='Bronze',
        marker=dict(symbol=3,
            line=marker_line, size=marker_size, color='rgb(205, 127, 50)', opacity=marker_opacity),
    )
)

fig.add_trace(
    go.Scatter(
        x = df_pfe['load_%'],
        y = df_pfe['efficiency_%'],
        mode='lines',
        name='PFE600',
        marker=dict(
            color='black', 
        )
    )
)

 # y axis title
ytitle = go.layout.Annotation(
        x=-0.01,
        y=1.15,
        xref="paper",
        yref="paper",
        text="Efficiency (%)",
        showarrow=False,
        xanchor='left'
    )

# Define the custom layout options
custom_layout = dict(
    xaxis=dict(
        title=dict(
            text='Power load (%)',
            font={'size':font_size_px}
        ),
        range=[-5,105],
    ),
    yaxis=dict(
        title=None,
        range=[80,100],
    ),
    legend=dict(
        orientation='v'
    ),
    annotations=[ytitle],
    margin=dict(l=0, r=0, t=25, b=0),
)
# Combine with the defaults and apply
layout = default_layout.copy()
layout.update(custom_layout)
fig.update_layout(layout)

fig.show()
fig.write_image(out_path/'80Plus-curves.pdf')

In [4]:
# ===============
# Interpolation functions for the different 80Plus standards
# ===============

# Function f(x) for 80PLUS Titanium standard
def t(x):
    const = 0.94 - eff_interp(0.20) + 0.02
    return eff_interp(x) + const
    # return -0.2083 * x**2 + 0.2125 * x + 0.90583

# Function f(x) for 80PLUS Platinum standard
def p(x):
    const = 0
    return eff_interp(x) + const
    # return -0.2417 * (x)**2 + 0.3025 * (x) + 0.84917

# Function f(x) for 80PLUS Gold standard
def g(x):
    const = 0.92 - eff_interp(0.50)
    return eff_interp(x) + const
    # return -0.2667 * (x)**2 + 0.32 * (x) + 0.8267

# Function f(x) for 80PLUS Silver standard
def s(x):
    const = 0.89 - eff_interp(0.50)
    return eff_interp(x) + const
    # return -0.2667 * (x)**2 + 0.32 * (x) + 0.7967

# Function f(x) for 80PLUS Bronze standard
def b(x):
    const = 0.85 - eff_interp(0.50)
    return eff_interp(x) + const
    # return -0.2667 * (x)**2 + 0.32 * (x) + 0.7567

# Function f(x) for an 80PLUS-like standard
# with custom constent term
def custom_eff(x,const):
    return eff_interp(x) + const

In [5]:
# ===============
# Load the PSU efficiency data
# ===============

def load_src_data():
        
    src_file = Path(input_path,'psu-efficiency.csv')
    df = pd.read_csv(src_file)

    df['load_PSU1'] = df['median_power_PSU1'].div(df['PSU_capacity'])
    df['load_PSU2'] = df['median_power_PSU2'].div(df['PSU_capacity'])

    return df

df = load_src_data()

df


,router_id,router_model,mean_power_PSU1,mean_power_PSU2,median_power_PSU1,median_power_PSU2,PSU_capacity,efficiency_PSU1,efficiency_PSU2,inlet_temperature,load_PSU1,load_PSU2
0,swiFN1,NCS-55A1-24H,191.171476,180.411720,191.020644,180.411000,1100,0.808791,0.839479,NaN,0.173655,0.164010
1,swiWY1,NCS-55A1-24H,194.310960,165.750012,194.311000,165.499000,1100,0.878525,0.922219,NaN,0.176646,0.150454
2,swiES3,NCS-55A1-24H,191.890173,166.704280,191.782000,164.063000,1100,0.938731,0.857143,NaN,0.174347,0.149148
3,swiUS1,NCS-55A1-24H,180.736509,167.841851,180.950930,167.873916,1100,0.872845,0.790150,NaN,0.164501,0.152613
4,swiTR3,NCS-55A1-24H,212.974344,180.248504,213.636000,180.261268,1100,0.883041,0.869099,NaN,0.194215,0.163874
...,...,...,...,...,...,...,...,...,...,...,...,...
102,swiWB1,NCS-55A1-48Q6H,186.161235,170.581386,185.614968,171.750000,1100,0.925602,0.812227,NaN,0.168741,0.156136
103,swiUN540,N540-24Z8Q2C-M,79.681341,79.645633,79.698000,79.665019,400,0.679551,0.693708,NaN,0.199245,0.199163
104,swiOR1,WS-C6504-E,164.589201,172.267520,164.781000,169.380210,2700,NaN,NaN,NaN,0.061030,0.062733
105,swiOT2,8201-24H8FH,110.206985,156.885401,112.000000,158.175901,2000,0.545455,0.645652,26.0,0.056000,0.079088


## Plot eff=f(load)

In [6]:
# ===============
# Plot the PSU conversion efficiency data
# ===============

df = load_src_data()

fig = go.Figure()

marker_opacity = 0.5
marker_size = 9
marker_border_size = 1
marker_line = dict(
    width=marker_border_size,
    color='black'
)

# Combine the data from both PSU
loads = df['load_PSU1'].tolist()
loads.append(df['load_PSU2'].tolist())
efficiencies = df['efficiency_PSU1'].tolist()
efficiencies.append(df['efficiency_PSU2'].tolist())


fig.add_trace(
    go.Scatter(
        x = df_pfe['load_%']/100,
        y = df_pfe['efficiency_%']/100 + 0.94 - eff_interp(0.20) + 0.02,
        mode='lines',
        name='Titanium',
        marker=dict(
            color='rgb(121, 121, 130)', 
        ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = [0.2],
        y = [0.96],
        mode='markers',
        name='Titanium',
        marker=dict(
            symbol=19, 
            line=marker_line,
            size=marker_size, 
            color='rgb(121, 121, 130)', 
            ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = df_pfe['load_%']/100,
        y = df_pfe['efficiency_%']/100,
        mode='lines',
        name='Platinum',
        marker=dict(
            color='rgb(209, 208, 206)', 
        ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = [0.2],
        y = [0.922],
        mode='markers',
        name='Platinum',
        marker=dict(symbol=0, 
            line=marker_line,size=marker_size, color='rgb(209, 208, 206)', ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = df_pfe['load_%']/100,
        y = df_pfe['efficiency_%']/100 + 0.92 - eff_interp(0.50),
        mode='lines',
        name='Gold',
        marker=dict(
            color='rgb(255, 215, 0)', 
        ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = [0.20],
        y = [0.899],
        mode='markers',
        name='Gold',
        marker=dict(symbol=2,
            line=marker_line, size=marker_size, color='rgb(255, 215, 0)', 
            ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = df_pfe['load_%']/100,
        y = df_pfe['efficiency_%']/100 + 0.89 - eff_interp(0.50),
        mode='lines',
        name='Silver',
        marker=dict(
            color='rgb(192, 192, 192)', 
        ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = [0.20],
        y = [0.87],
        mode='markers',
        name='Silver',
        marker=dict(symbol=4,
            line=marker_line, size=marker_size, color='rgb(192, 192, 192)', 
            ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = df_pfe['load_%']/100,
        y = df_pfe['efficiency_%']/100 + 0.85 - eff_interp(0.50),
        mode='lines',
        name='Bronze',
        marker=dict(
            color='rgb(205, 127, 50)', 
        ),
        showlegend=False
    )
)
fig.add_trace(
    go.Scatter(
        x = [0.20],
        y = [0.83],
        mode='markers',
        name='Bronze',
        marker=dict(symbol=3,
            line=marker_line, size=marker_size, color='rgb(205, 127, 50)', 
            ),
        showlegend=False
    )
)


fig.add_trace(
    go.Scatter(
        x = loads,
        y = efficiencies,
        mode='markers',
        marker=dict(
            opacity=marker_opacity,
        ),
        showlegend=False
    )
)


 # y axis title
ytitle = go.layout.Annotation(
        x=-0.01,
        y=1.15,
        xref="paper",
        yref="paper",
        text="Efficiency (%)",
        showarrow=False,
        xanchor='left'
    )

# Define the custom layout options
custom_layout = dict(
    xaxis=dict(
        title=dict(
            text='Power load (%)',
            font={'size':font_size_px}
        ),
        range=[0.03,0.23],
        tickmode = 'array',
        tickvals = [0.05, 0.1, 0.15, 0.2],
        ticktext = [5, 10, 15, 20]
    ),
    yaxis=dict(
        title=None,
        range=[0.50,1.05],
    ),
    legend=dict(
        orientation='v'
    ),
    width=0.3*landscapewidth_px,
    annotations=[ytitle],
    margin=dict(l=0, r=0, t=25, b=0),
)
# Combine with the defaults and apply
layout = default_layout.copy()
layout.update(custom_layout)
fig.update_layout(layout)


fig.show()
fig.write_image(out_path/'efficiency_all.pdf')

In [7]:
# ===============
# Highlight the data from a given router model
# ===============


# Filter for the router model of interest

# model_id = '8201-32FH'
# model_id = 'NCS-55A1-24H'
model_id = 'ASR-920-24SZ-M'
# model_id = '8201-24H8FH'
tmp = df.loc[df['router_model']== model_id]

# Combine the data from both PSU
loads = tmp['load_PSU1'].tolist()
loads.append(tmp['load_PSU2'].tolist())
efficiencies = tmp['efficiency_PSU1'].tolist()
efficiencies.append(tmp['efficiency_PSU2'].tolist())

fig.add_trace(
    go.Scatter(
        x = loads,
        y = efficiencies,
        mode='markers',
        marker=dict(
            color='red', 
            opacity=marker_opacity,
        ),
        showlegend=False
    )
)

# Define the custom layout options
custom_layout = dict(
    width=0.2*landscapewidth_px,
)
# Combine with the defaults and apply
# layout = default_layout.copy() <- Don't copy! We want to update from the previous plot
layout.update(custom_layout)
fig.update_layout(layout)

fig.write_image(out_path/str('efficiency_'+model_id+'.pdf'))

fig.show()

## How much would we save with better PSUs?

In [8]:
df = load_src_data()
tex_output = ''

# Compute the theoretic efficiency for different standards
for standard in [b,s,g,p,t]:
    for PSU in ['1','2']:
        # .. compute the efficiency
        label_eff = 'efficiency_PSU' + PSU + '_' + standard.__name__
        df[label_eff] = df['load_PSU' + PSU].apply(standard)
        df[label_eff] = df[[label_eff,'efficiency_PSU' + PSU]].max(axis=1)
        
        # .. derive the resulting power draw
        label_power = 'median_power_PSU' + PSU + '_' + standard.__name__
        df[label_power] = df['median_power_PSU' + PSU] * df['efficiency_PSU' + PSU] / df[label_eff]

        # .. compute the correspinding savings
        label_savings_abs = 'savings_abs_PSU' + PSU + '_' + standard.__name__
        label_savings_rel = 'savings_rel_PSU' + PSU + '_' + standard.__name__
        df[label_savings_abs] = df['median_power_PSU' + PSU] - df[label_power] 
        df[label_savings_rel] = df[label_savings_abs] / df['median_power_PSU' + PSU] 

    # .. Compute the gains
    savings_total_abs = df['savings_abs_PSU1_'+standard.__name__].sum(axis=0) +  df['savings_abs_PSU2_'+standard.__name__].sum(axis=0) 
    savings_total_rel = savings_total_abs / (df['median_power_PSU1'].sum(axis=0) + df['median_power_PSU2'].sum(axis=0))

    print('For',standard.__name__)
    print("{}\\% ({} W)\n".format(
        int(100*savings_total_rel),
        int(savings_total_abs)
        ))
    
    tex_output += '& {}\\% ({} W)'.format(
        int(100*savings_total_rel),
        int(savings_total_abs)
        )
tex_output += '\\\\'
df

print(tex_output)

For b
2\% (482 W)

For s
3\% (737 W)

For g
4\% (958 W)

For p
5\% (1156 W)

For t
7\% (1563 W)

& 2\% (482 W)& 3\% (737 W)& 4\% (958 W)& 5\% (1156 W)& 7\% (1563 W)\\


## How much would we save with better-sized PSUs?

In [9]:
df = load_src_data()
psu_capacities = sorted(df['PSU_capacity'].unique())

pd.set_option('display.max_columns', None)

overprovision_factor = [1,2,3]

def set_min_capacity(row, capacity_options, overprovision_factor):
    for c in sorted(capacity_options):
        if ((row['median_power_PSU1'] > c/overprovision_factor) or
            (row['median_power_PSU2'] > c/overprovision_factor)):
            continue
        else:
            return c
        
def set_new_capacity(row, capacity):
    if (row['min_capacity'] > capacity):
        return row['min_capacity']
    else:
        return capacity
    
def get_new_efficiency(row, PSU, label_load):
    offset = row['efficiency_PSU' + PSU] - eff_interp(row['load_PSU' + PSU])
    return custom_eff(row[label_load],offset)
    
for k in overprovision_factor:

    tex_output = ''
    print('k = {}'.format(k))
    
    # .. define the minimal capacity required per router
    df['min_capacity'] = df.apply(set_min_capacity, args=(psu_capacities, k), axis=1)

    # .. set the different cases of smallest considered capacities
    for capacity in psu_capacities:

        # .. compute the min capacity per router per case
        label_capacity = 'capacity_'+str(capacity)+'+'
        df[label_capacity] = df.apply(set_new_capacity, args=(capacity, ), axis=1)
        
        for PSU in ['1','2']:
            # .. compute the corresponding load
            label_load = 'load_PSU' + PSU + '_' + label_capacity
            df[label_load] = df['median_power_PSU' + PSU] / df[label_capacity]
            # .. compute the corresponding efficiency
            label_eff = 'efficiency_PSU' + PSU + '_' + label_capacity
            df[label_eff] = df.apply(get_new_efficiency, args=(PSU,label_load,), axis=1)

            # .. derive the resulting power draw
            label_power = 'median_power_PSU' + PSU + '_' + label_capacity
            df[label_power] = df['median_power_PSU' + PSU] * df['efficiency_PSU' + PSU] / df[label_eff]

            # .. compute the correspinding savings
            label_savings_abs = 'savings_abs_PSU' + PSU + '_' + label_capacity
            label_savings_rel = 'savings_rel_PSU' + PSU + '_' + label_capacity
            df[label_savings_abs] = df['median_power_PSU' + PSU] - df[label_power] 
            df[label_savings_rel] = df[label_savings_abs] / df['median_power_PSU' + PSU] 



        # .. Compute the total gains
        savings_total_abs = df['savings_abs_PSU1_' + label_capacity].sum(axis=0) +  df['savings_abs_PSU2_' + label_capacity].sum(axis=0) 
        savings_total_rel = savings_total_abs / (df['median_power_PSU1'].sum(axis=0) + df['median_power_PSU2'].sum(axis=0))

        print('For $',label_capacity,'$')
        print("{}\\% ({} W)\n".format(
            int(savings_total_rel*100),
            int(savings_total_abs)
            ))
        

        tex_output += '& {}\\% ({} W)'.format(
                int(100*savings_total_rel),
                int(savings_total_abs)
                )

    tex_output += '\\\\'
    print(tex_output)

    display(df)

k = 1
For $ capacity_250+ $
2\% (520 W)

For $ capacity_400+ $
2\% (456 W)

For $ capacity_750+ $
1\% (287 W)

For $ capacity_1100+ $
0\% (-21 W)

For $ capacity_2000+ $
-1\% (-247 W)

For $ capacity_2700+ $
-1\% (-247 W)

& 2\% (520 W)& 2\% (456 W)& 1\% (287 W)& 0\% (-21 W)& -1\% (-247 W)& -1\% (-247 W)\\


,router_id,router_model,mean_power_PSU1,mean_power_PSU2,median_power_PSU1,median_power_PSU2,PSU_capacity,efficiency_PSU1,efficiency_PSU2,inlet_temperature,load_PSU1,load_PSU2,min_capacity,capacity_250+,load_PSU1_capacity_250+,efficiency_PSU1_capacity_250+,median_power_PSU1_capacity_250+,savings_abs_PSU1_capacity_250+,savings_rel_PSU1_capacity_250+,load_PSU2_capacity_250+,efficiency_PSU2_capacity_250+,median_power_PSU2_capacity_250+,savings_abs_PSU2_capacity_250+,savings_rel_PSU2_capacity_250+,capacity_400+,load_PSU1_capacity_400+,efficiency_PSU1_capacity_400+,median_power_PSU1_capacity_400+,savings_abs_PSU1_capacity_400+,savings_rel_PSU1_capacity_400+,load_PSU2_capacity_400+,efficiency_PSU2_capacity_400+,median_power_PSU2_capacity_400+,savings_abs_PSU2_capacity_400+,savings_rel_PSU2_capacity_400+,capacity_750+,load_PSU1_capacity_750+,efficiency_PSU1_capacity_750+,median_power_PSU1_capacity_750+,savings_abs_PSU1_capacity_750+,savings_rel_PSU1_capacity_750+,load_PSU2_capacity_750+,efficiency_PSU2_capacity_750+,median_power_PSU2_capacity_750+,savings_abs_PSU2_capacity_750+,savings_rel_PSU2_capacity_750+,capacity_1100+,load_PSU1_capacity_1100+,efficiency_PSU1_capacity_1100+,median_power_PSU1_capacity_1100+,savings_abs_PSU1_capacity_1100+,savings_rel_PSU1_capacity_1100+,load_PSU2_capacity_1100+,efficiency_PSU2_capacity_1100+,median_power_PSU2_capacity_1100+,savings_abs_PSU2_capacity_1100+,savings_rel_PSU2_capacity_1100+,capacity_2000+,load_PSU1_capacity_2000+,efficiency_PSU1_capacity_2000+,median_power_PSU1_capacity_2000+,savings_abs_PSU1_capacity_2000+,savings_rel_PSU1_capacity_2000+,load_PSU2_capacity_2000+,efficiency_PSU2_capacity_2000+,median_power_PSU2_capacity_2000+,savings_abs_PSU2_capacity_2000+,savings_rel_PSU2_capacity_2000+,capacity_2700+,load_PSU1_capacity_2700+,efficiency_PSU1_capacity_2700+,median_power_PSU1_capacity_2700+,savings_abs_PSU1_capacity_2700+,savings_rel_PSU1_capacity_2700+,load_PSU2_capacity_2700+,efficiency_PSU2_capacity_2700+,median_power_PSU2_capacity_2700+,savings_abs_PSU2_capacity_2700+,savings_rel_PSU2_capacity_2700+
0,swiFN1,NCS-55A1-24H,191.171476,180.411720,191.020644,180.411000,1100,0.808791,0.839479,NaN,0.173655,0.164010,250,250,0.764083,0.838366,184.282044,6.738600,0.035277,0.721644,0.875435,173.001283,7.409717,0.041071,400,0.477552,0.842031,183.479993,7.540651,0.039476,0.451028,0.877356,172.622437,7.788563,0.043171,750,0.254694,0.831450,185.814847,5.205797,0.027253,0.240548,0.865711,174.944344,5.466656,0.030301,1100,0.173655,0.808791,191.020644,0.000000e+00,0.000000e+00,0.164010,0.839479,180.411000,0.000000,0.000000,2000,0.095510,0.780121,198.040927,-7.020283,-0.036751,0.090205,0.815900,185.624888,-5.213888,-0.028900,2700,0.070748,0.780121,198.040927,-7.020283,-0.036751,0.066819,0.815900,185.624888,-5.213888,-0.028900
1,swiWY1,NCS-55A1-24H,194.310960,165.750012,194.311000,165.499000,1100,0.878525,0.922219,NaN,0.176646,0.150454,250,250,0.777244,0.906124,188.392690,5.918310,0.030458,0.661996,0.966849,157.859465,7.639535,0.046161,400,0.485778,0.910392,187.509455,6.801545,0.035003,0.413747,0.966303,157.948688,7.550312,0.045621,750,0.259081,0.900137,189.645608,4.665392,0.024010,0.220665,0.952680,160.207275,5.291725,0.031974,1100,0.176646,0.878525,194.311000,0.000000e+00,0.000000e+00,0.150454,0.922219,165.499000,0.000000,0.000000,2000,0.097156,0.848341,201.224651,-6.913651,-0.035580,0.082750,0.905839,168.491656,-2.992656,-0.018083,2700,0.071967,0.848341,201.224651,-6.913651,-0.035580,0.061296,0.905839,168.491656,-2.992656,-0.018083
2,swiES3,NCS-55A1-24H,191.890173,166.704280,191.782000,164.063000,1100,0.938731,0.857143,NaN,0.174347,0.149148,250,250,0.767128,0.967833,186.015160,5.766840,0.030070,0.656252,0.902584,155.803079,8.259921,0.050346,400,0.479455,0.971638,185.286829,6.495171,0.033867,0.410157,0.901826,155.934148,8.128852,0.049547,750,0.255709,0.961134,187.311819,4.470181,0.023309,0.218751,0.887972,158.366918,5.696082,0.034719,1100,0.174347,0.938731,191.782000,0.000000e+00,0.000000e+00,0.1491

k = 2
For $ capacity_250+ $
2\% (502 W)

For $ capacity_400+ $
2\% (432 W)

For $ capacity_750+ $
1\% (287 W)

For $ capacity_1100+ $
0\% (-21 W)

For $ capacity_2000+ $
-1\% (-247 W)

For $ capacity_2700+ $
-1\% (-247 W)

& 2\% (502 W)& 2\% (432 W)& 1\% (287 W)& 0\% (-21 W)& -1\% (-247 W)& -1\% (-247 W)\\


,router_id,router_model,mean_power_PSU1,mean_power_PSU2,median_power_PSU1,median_power_PSU2,PSU_capacity,efficiency_PSU1,efficiency_PSU2,inlet_temperature,load_PSU1,load_PSU2,min_capacity,capacity_250+,load_PSU1_capacity_250+,efficiency_PSU1_capacity_250+,median_power_PSU1_capacity_250+,savings_abs_PSU1_capacity_250+,savings_rel_PSU1_capacity_250+,load_PSU2_capacity_250+,efficiency_PSU2_capacity_250+,median_power_PSU2_capacity_250+,savings_abs_PSU2_capacity_250+,savings_rel_PSU2_capacity_250+,capacity_400+,load_PSU1_capacity_400+,efficiency_PSU1_capacity_400+,median_power_PSU1_capacity_400+,savings_abs_PSU1_capacity_400+,savings_rel_PSU1_capacity_400+,load_PSU2_capacity_400+,efficiency_PSU2_capacity_400+,median_power_PSU2_capacity_400+,savings_abs_PSU2_capacity_400+,savings_rel_PSU2_capacity_400+,capacity_750+,load_PSU1_capacity_750+,efficiency_PSU1_capacity_750+,median_power_PSU1_capacity_750+,savings_abs_PSU1_capacity_750+,savings_rel_PSU1_capacity_750+,load_PSU2_capacity_750+,efficiency_PSU2_capacity_750+,median_power_PSU2_capacity_750+,savings_abs_PSU2_capacity_750+,savings_rel_PSU2_capacity_750+,capacity_1100+,load_PSU1_capacity_1100+,efficiency_PSU1_capacity_1100+,median_power_PSU1_capacity_1100+,savings_abs_PSU1_capacity_1100+,savings_rel_PSU1_capacity_1100+,load_PSU2_capacity_1100+,efficiency_PSU2_capacity_1100+,median_power_PSU2_capacity_1100+,savings_abs_PSU2_capacity_1100+,savings_rel_PSU2_capacity_1100+,capacity_2000+,load_PSU1_capacity_2000+,efficiency_PSU1_capacity_2000+,median_power_PSU1_capacity_2000+,savings_abs_PSU1_capacity_2000+,savings_rel_PSU1_capacity_2000+,load_PSU2_capacity_2000+,efficiency_PSU2_capacity_2000+,median_power_PSU2_capacity_2000+,savings_abs_PSU2_capacity_2000+,savings_rel_PSU2_capacity_2000+,capacity_2700+,load_PSU1_capacity_2700+,efficiency_PSU1_capacity_2700+,median_power_PSU1_capacity_2700+,savings_abs_PSU1_capacity_2700+,savings_rel_PSU1_capacity_2700+,load_PSU2_capacity_2700+,efficiency_PSU2_capacity_2700+,median_power_PSU2_capacity_2700+,savings_abs_PSU2_capacity_2700+,savings_rel_PSU2_capacity_2700+
0,swiFN1,NCS-55A1-24H,191.171476,180.411720,191.020644,180.411000,1100,0.808791,0.839479,NaN,0.173655,0.164010,400,400,0.477552,0.842031,183.479993,7.540651,0.039476,0.451028,0.877356,172.622437,7.788563,0.043171,400,0.477552,0.842031,183.479993,7.540651,0.039476,0.451028,0.877356,172.622437,7.788563,0.043171,750,0.254694,0.831450,185.814847,5.205797,0.027253,0.240548,0.865711,174.944344,5.466656,0.030301,1100,0.173655,0.808791,191.020644,0.000000e+00,0.000000e+00,0.164010,0.839479,180.411000,0.000000,0.000000,2000,0.095510,0.780121,198.040927,-7.020283,-0.036751,0.090205,0.815900,185.624888,-5.213888,-0.028900,2700,0.070748,0.780121,198.040927,-7.020283,-0.036751,0.066819,0.815900,185.624888,-5.213888,-0.028900
1,swiWY1,NCS-55A1-24H,194.310960,165.750012,194.311000,165.499000,1100,0.878525,0.922219,NaN,0.176646,0.150454,400,400,0.485778,0.910392,187.509455,6.801545,0.035003,0.413747,0.966303,157.948688,7.550312,0.045621,400,0.485778,0.910392,187.509455,6.801545,0.035003,0.413747,0.966303,157.948688,7.550312,0.045621,750,0.259081,0.900137,189.645608,4.665392,0.024010,0.220665,0.952680,160.207275,5.291725,0.031974,1100,0.176646,0.878525,194.311000,0.000000e+00,0.000000e+00,0.150454,0.922219,165.499000,0.000000,0.000000,2000,0.097156,0.848341,201.224651,-6.913651,-0.035580,0.082750,0.905839,168.491656,-2.992656,-0.018083,2700,0.071967,0.848341,201.224651,-6.913651,-0.035580,0.061296,0.905839,168.491656,-2.992656,-0.018083
2,swiES3,NCS-55A1-24H,191.890173,166.704280,191.782000,164.063000,1100,0.938731,0.857143,NaN,0.174347,0.149148,400,400,0.479455,0.971638,185.286829,6.495171,0.033867,0.410157,0.901826,155.934148,8.128852,0.049547,400,0.479455,0.971638,185.286829,6.495171,0.033867,0.410157,0.901826,155.934148,8.128852,0.049547,750,0.255709,0.961134,187.311819,4.470181,0.023309,0.218751,0.887972,158.366918,5.696082,0.034719,1100,0.174347,0.938731,191.782000,0.000000e+00,0.000000e+00,0.1491

k = 3
For $ capacity_250+ $
1\% (367 W)

For $ capacity_400+ $
1\% (298 W)

For $ capacity_750+ $
1\% (287 W)

For $ capacity_1100+ $
0\% (-21 W)

For $ capacity_2000+ $
-1\% (-247 W)

For $ capacity_2700+ $
-1\% (-247 W)

& 1\% (367 W)& 1\% (298 W)& 1\% (287 W)& 0\% (-21 W)& -1\% (-247 W)& -1\% (-247 W)\\


,router_id,router_model,mean_power_PSU1,mean_power_PSU2,median_power_PSU1,median_power_PSU2,PSU_capacity,efficiency_PSU1,efficiency_PSU2,inlet_temperature,load_PSU1,load_PSU2,min_capacity,capacity_250+,load_PSU1_capacity_250+,efficiency_PSU1_capacity_250+,median_power_PSU1_capacity_250+,savings_abs_PSU1_capacity_250+,savings_rel_PSU1_capacity_250+,load_PSU2_capacity_250+,efficiency_PSU2_capacity_250+,median_power_PSU2_capacity_250+,savings_abs_PSU2_capacity_250+,savings_rel_PSU2_capacity_250+,capacity_400+,load_PSU1_capacity_400+,efficiency_PSU1_capacity_400+,median_power_PSU1_capacity_400+,savings_abs_PSU1_capacity_400+,savings_rel_PSU1_capacity_400+,load_PSU2_capacity_400+,efficiency_PSU2_capacity_400+,median_power_PSU2_capacity_400+,savings_abs_PSU2_capacity_400+,savings_rel_PSU2_capacity_400+,capacity_750+,load_PSU1_capacity_750+,efficiency_PSU1_capacity_750+,median_power_PSU1_capacity_750+,savings_abs_PSU1_capacity_750+,savings_rel_PSU1_capacity_750+,load_PSU2_capacity_750+,efficiency_PSU2_capacity_750+,median_power_PSU2_capacity_750+,savings_abs_PSU2_capacity_750+,savings_rel_PSU2_capacity_750+,capacity_1100+,load_PSU1_capacity_1100+,efficiency_PSU1_capacity_1100+,median_power_PSU1_capacity_1100+,savings_abs_PSU1_capacity_1100+,savings_rel_PSU1_capacity_1100+,load_PSU2_capacity_1100+,efficiency_PSU2_capacity_1100+,median_power_PSU2_capacity_1100+,savings_abs_PSU2_capacity_1100+,savings_rel_PSU2_capacity_1100+,capacity_2000+,load_PSU1_capacity_2000+,efficiency_PSU1_capacity_2000+,median_power_PSU1_capacity_2000+,savings_abs_PSU1_capacity_2000+,savings_rel_PSU1_capacity_2000+,load_PSU2_capacity_2000+,efficiency_PSU2_capacity_2000+,median_power_PSU2_capacity_2000+,savings_abs_PSU2_capacity_2000+,savings_rel_PSU2_capacity_2000+,capacity_2700+,load_PSU1_capacity_2700+,efficiency_PSU1_capacity_2700+,median_power_PSU1_capacity_2700+,savings_abs_PSU1_capacity_2700+,savings_rel_PSU1_capacity_2700+,load_PSU2_capacity_2700+,efficiency_PSU2_capacity_2700+,median_power_PSU2_capacity_2700+,savings_abs_PSU2_capacity_2700+,savings_rel_PSU2_capacity_2700+
0,swiFN1,NCS-55A1-24H,191.171476,180.411720,191.020644,180.411000,1100,0.808791,0.839479,NaN,0.173655,0.164010,750,750,0.254694,0.831450,185.814847,5.205797,0.027253,0.240548,0.865711,174.944344,5.466656,0.030301,750,0.254694,0.831450,185.814847,5.205797,0.027253,0.240548,0.865711,174.944344,5.466656,0.030301,750,0.254694,0.831450,185.814847,5.205797,0.027253,0.240548,0.865711,174.944344,5.466656,0.030301,1100,0.173655,0.808791,191.020644,0.000000e+00,0.000000e+00,0.164010,0.839479,180.411000,0.000000,0.000000,2000,0.095510,0.780121,198.040927,-7.020283,-0.036751,0.090205,0.815900,185.624888,-5.213888,-0.028900,2700,0.070748,0.780121,198.040927,-7.020283,-0.036751,0.066819,0.815900,185.624888,-5.213888,-0.028900
1,swiWY1,NCS-55A1-24H,194.310960,165.750012,194.311000,165.499000,1100,0.878525,0.922219,NaN,0.176646,0.150454,750,750,0.259081,0.900137,189.645608,4.665392,0.024010,0.220665,0.952680,160.207275,5.291725,0.031974,750,0.259081,0.900137,189.645608,4.665392,0.024010,0.220665,0.952680,160.207275,5.291725,0.031974,750,0.259081,0.900137,189.645608,4.665392,0.024010,0.220665,0.952680,160.207275,5.291725,0.031974,1100,0.176646,0.878525,194.311000,0.000000e+00,0.000000e+00,0.150454,0.922219,165.499000,0.000000,0.000000,2000,0.097156,0.848341,201.224651,-6.913651,-0.035580,0.082750,0.905839,168.491656,-2.992656,-0.018083,2700,0.071967,0.848341,201.224651,-6.913651,-0.035580,0.061296,0.905839,168.491656,-2.992656,-0.018083
2,swiES3,NCS-55A1-24H,191.890173,166.704280,191.782000,164.063000,1100,0.938731,0.857143,NaN,0.174347,0.149148,750,750,0.255709,0.961134,187.311819,4.470181,0.023309,0.218751,0.887972,158.366918,5.696082,0.034719,750,0.255709,0.961134,187.311819,4.470181,0.023309,0.218751,0.887972,158.366918,5.696082,0.034719,750,0.255709,0.961134,187.311819,4.470181,0.023309,0.218751,0.887972,158.366918,5.696082,0.034719,1100,0.174347,0.938731,191.782000,0.000000e+00,0.000000e+00,0.1491

## How much would we save by loading only one PSU instead of two?

In [10]:
# Load the data
df = load_src_data()
tex_output = ''

# .. compute the total load
df['total_power_out'] = df['median_power_PSU1']*df['efficiency_PSU1'] + df['median_power_PSU2']*df['efficiency_PSU2']
df['total_load'] = df['total_power_out'].div(df['PSU_capacity'])

def get_new_efficiency(row, PSU, label_load):
    offset = row['efficiency_PSU' + PSU] - eff_interp(row['load_PSU' + PSU])
    return custom_eff(row[label_load],offset)

# .. compute the efficiency for that load if running on only one of the PSUs
for PSU in ['1','2']:
    label_eff = 'efficiency_PSU' + PSU + '_total_load' 
    df[label_eff] = df.apply(get_new_efficiency, args=(PSU,'total_load',), axis=1)
# .. take the max
df['max_efficiency'] = df[['efficiency_PSU1_total_load','efficiency_PSU2_total_load']].max(axis=1)
# .. get the resulting total power in
df['total_power_in'] = df['total_power_out'] / df['max_efficiency']

# .. compute the savings per router
df['savings_abs'] = (df['median_power_PSU1']+ df['median_power_PSU2']) - df['total_power_in']
df['savings_rel'] = df['savings_abs'] / (df['median_power_PSU1']+ df['median_power_PSU2'])

# .. compute the global savings
savings_total_abs = df['savings_abs'].sum(axis=0)
savings_total_rel = savings_total_abs / (df['median_power_PSU1'].sum(axis=0) + df['median_power_PSU2'].sum(axis=0))


print('For using only one PSU')
print("{}\\% ({} W)\n".format(
    int(savings_total_rel*100),
    int(savings_total_abs)
    ))

tex_output += '& {}\\% ({} W)'.format(
        int(100*savings_total_rel),
        int(savings_total_abs)
        )

print(tex_output)

df

For using only one PSU
4\% (1002 W)

& 4\% (1002 W)


,router_id,router_model,mean_power_PSU1,mean_power_PSU2,median_power_PSU1,median_power_PSU2,PSU_capacity,efficiency_PSU1,efficiency_PSU2,inlet_temperature,load_PSU1,load_PSU2,total_power_out,total_load,efficiency_PSU1_total_load,efficiency_PSU2_total_load,max_efficiency,total_power_in,savings_abs,savings_rel
0,swiFN1,NCS-55A1-24H,191.171476,180.411720,191.020644,180.411000,1100,0.808791,0.839479,NaN,0.173655,0.164010,305.947134,0.278134,0.833595,0.869374,0.869374,351.916615,19.515029,0.052540
1,swiWY1,NCS-55A1-24H,194.310960,165.750012,194.311000,165.499000,1100,0.878525,0.922219,NaN,0.176646,0.150454,323.333346,0.293939,0.903119,0.960617,0.960617,336.589090,23.220910,0.064537
2,swiES3,NCS-55A1-24H,191.890173,166.704280,191.782000,164.063000,1100,0.938731,0.857143,NaN,0.174347,0.149148,320.657109,0.291506,0.964286,0.896049,0.964286,332.533163,23.311837,0.065511
3,swiUS1,NCS-55A1-24H,180.736509,167.841851,180.950930,167.873916,1100,0.872845,0.790150,NaN,0.164501,0.152613,290.587640,0.264171,0.901251,0.824865,0.901251,322.427059,26.397787,0.075676
4,swiTR3,NCS-55A1-24H,212.974344,180.248504,213.636000,180.261268,1100,0.883041,0.869099,NaN,0.194215,0.163874,345.314169,0.313922,0.900745,0.901899,0.901899,382.874613,11.022655,0.027984
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,swiWB1,NCS-55A1-48Q6H,186.161235,170.581386,185.614968,171.750000,1100,0.925602,0.812227,NaN,0.168741,0.156136,311.305539,0.283005,0.953428,0.846724,0.953428,326.511806,30.853161,0.086335
103,swiUN540,N540-24Z8Q2C-M,79.681341,79.645633,79.698000,79.665019,400,0.679551,0.693708,NaN,0.199245,0.199163,109.423071,0.273558,0.691571,0.705768,0.705768,155.041145,4.321874,0.027120
104,swiOR1,WS-C6504-E,164.589201,172.267520,164.781000,169.380210,2700,NaN,NaN,NaN,0.061030,0.062733,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
105,swiOT2,8201-24H8FH,110.206985,156.885401,112.000000,158.175901,2000,0.545455,0.645652,26.0,0.056000,0.079088,163.217524,0.081609,0.545455,0.645652,0.645652,252.794818,17.381084,0.064332


## How much would we save by loading only one PSU instead of two AND that one be of better quality?

In [11]:
# [continuing from the previous df]
tex_output = ''

# Compute the theoretic efficiency for different standards
for standard in [b,s,g,p,t]:
    # .. compute the efficiency
    label_eff = 'efficiency_' + standard.__name__
    df[label_eff] = df['total_load'].apply(standard)
    df[label_eff] = df[[label_eff,'max_efficiency']].max(axis=1)

    # .. derive the resulting power draw
    label_power = 'median_power_' + standard.__name__
    df[label_power] = df['total_power_in'] * df['max_efficiency'] / df[label_eff]

    # .. compute the savings per router
    label_savings_abs = 'savings_abs_' + standard.__name__
    label_savings_rel = 'savings_rel_' + standard.__name__
    df[label_savings_abs] = df['total_power_in'] - df[label_power] 
    df[label_savings_rel] = df[label_savings_abs] / df['total_power_in'] 

    # .. compute the global savings
    print('For using only one PSU of at least standard ',standard.__name__, '\\\\')

    # .. 1. from the first step
    savings_total_abs = df['savings_abs'].sum(axis=0)
    savings_total_rel = savings_total_abs / (df['median_power_PSU1'].sum(axis=0) + df['median_power_PSU2'].sum(axis=0))
    print("using only one {}\\% ({} W)\\\\".format(
    int(100*savings_total_rel),
    int(savings_total_abs)
    ))
    
    # .. 2. from this step alone
    savings_total_abs = df['savings_abs_'+standard.__name__].sum(axis=0)
    savings_total_rel = savings_total_abs / (df['median_power_PSU1'].sum(axis=0) + df['median_power_PSU2'].sum(axis=0))
    print("using a better one {}\\% ({} W)\\\\".format(
    int(100*savings_total_rel),
    int(savings_total_abs)
    ))

    # .. 3. all together
    savings_total_abs = df['savings_abs_'+standard.__name__].sum(axis=0) + df['savings_abs'].sum(axis=0)
    savings_total_rel = savings_total_abs / (df['median_power_PSU1'].sum(axis=0) + df['median_power_PSU2'].sum(axis=0))
    print("total: {}\\% ({} W)\n".format(
    int(100*savings_total_rel),
    int(savings_total_abs)
    ))

    tex_output += '& {}\\% ({} W)'.format(
        int(100*savings_total_rel),
        int(savings_total_abs)
        )
    
tex_output += '\\\\'
print(tex_output)

df


For using only one PSU of at least standard  b \\
using only one 4\% (1002 W)\\
using a better one 1\% (237 W)\\
total: 5\% (1240 W)

For using only one PSU of at least standard  s \\
using only one 4\% (1002 W)\\
using a better one 1\% (389 W)\\
total: 6\% (1392 W)

For using only one PSU of at least standard  g \\
using only one 4\% (1002 W)\\
using a better one 2\% (525 W)\\
total: 7\% (1528 W)

For using only one PSU of at least standard  p \\
using only one 4\% (1002 W)\\
using a better one 3\% (657 W)\\
total: 7\% (1660 W)

For using only one PSU of at least standard  t \\
using only one 4\% (1002 W)\\
using a better one 4\% (971 W)\\
total: 9\% (1974 W)

& 5\% (1240 W)& 6\% (1392 W)& 7\% (1528 W)& 7\% (1660 W)& 9\% (1974 W)\\


,router_id,router_model,mean_power_PSU1,mean_power_PSU2,median_power_PSU1,median_power_PSU2,PSU_capacity,efficiency_PSU1,efficiency_PSU2,inlet_temperature,load_PSU1,load_PSU2,total_power_out,total_load,efficiency_PSU1_total_load,efficiency_PSU2_total_load,max_efficiency,total_power_in,savings_abs,savings_rel,efficiency_b,median_power_b,savings_abs_b,savings_rel_b,efficiency_s,median_power_s,savings_abs_s,savings_rel_s,efficiency_g,median_power_g,savings_abs_g,savings_rel_g,efficiency_p,median_power_p,savings_abs_p,savings_rel_p,efficiency_t,median_power_t,savings_abs_t,savings_rel_t
0,swiFN1,NCS-55A1-24H,191.171476,180.411720,191.020644,180.411000,1100,0.808791,0.839479,NaN,0.173655,0.164010,305.947134,0.278134,0.833595,0.869374,0.869374,351.916615,19.515029,0.052540,0.869374,351.916615,0.000000,0.000000,0.881300,347.154493,4.762122,0.013532,0.911300,335.726162,16.190454,0.046007,0.933872,327.611285,24.305331,0.069066,0.972056,314.742420,37.174195,0.105634
1,swiWY1,NCS-55A1-24H,194.310960,165.750012,194.311000,165.499000,1100,0.878525,0.922219,NaN,0.176646,0.150454,323.333346,0.293939,0.903119,0.960617,0.960617,336.589090,23.220910,0.064537,0.960617,336.589090,0.000000,0.000000,0.960617,336.589090,0.000000,0.000000,0.960617,336.589090,0.000000,0.000000,0.960617,336.589090,0.000000,0.000000,0.973360,332.182659,4.406431,0.013091
2,swiES3,NCS-55A1-24H,191.890173,166.704280,191.782000,164.063000,1100,0.938731,0.857143,NaN,0.174347,0.149148,320.657109,0.291506,0.964286,0.896049,0.964286,332.533163,23.311837,0.065511,0.964286,332.533163,0.000000,0.000000,0.964286,332.533163,0.000000,0.000000,0.964286,332.533163,0.000000,0.000000,0.964286,332.533163,0.000000,0.000000,0.973173,329.496631,3.036532,0.009132
3,swiUS1,NCS-55A1-24H,180.736509,167.841851,180.950930,167.873916,1100,0.872845,0.790150,NaN,0.164501,0.152613,290.587640,0.264171,0.901251,0.824865,0.901251,322.427059,26.397787,0.075676,0.901251,322.427059,0.000000,0.000000,0.901251,322.427059,0.000000,0.000000,0.910070,319.302464,3.124595,0.009691,0.932643,311.574391,10.852667,0.033659,0.970826,299.319970,23.107088,0.071666
4,swiTR3,NCS-55A1-24H,212.974344,180.248504,213.636000,180.261268,1100,0.883041,0.869099,NaN,0.194215,0.163874,345.314169,0.313922,0.900745,0.901899,0.901899,382.874613,11.022655,0.027984,0.901899,382.874613,0.000000,0.000000,0.901899,382.874613,0.000000,0.000000,0.914133,377.750310,5.124303,0.013384,0.936706,368.647285,14.227328,0.037159,0.974889,354.208581,28.666032,0.074871
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,swiWB1,NCS-55A1-48Q6H,186.161235,170.581386,185.614968,171.750000,1100,0.925602,0.812227,NaN,0.168741,0.156136,311.305539,0.283005,0.953428,0.846724,0.953428,326.511806,30.853161,0.086335,0.953428,326.511806,0.000000,0.000000,0.953428,326.511806,0.000000,0.000000,0.953428,326.511806,0.000000,0.000000,0.953428,326.511806,0.000000,0.000000,0.972485,320.113612,6.398194,0.019596
103,swiUN540,N540-24Z8Q2C-M,79.681341,79.645633,79.698000,79.665019,400,0.679551,0.693708,NaN,0.199245,0.199163,109.423071,0.273558,0.691571,0.705768,0.705768,155.041145,4.321874,0.027120,0.840897,130.126648,24.914496,0.160696,0.880897,124.217822,30.823323,0.198807,0.910897,120.126760,34.914385,0.225194,0.933469,117.221909,37.819236,0.243930,0.971653,112.615418,42.425727,0.273642
104,swiOR1,WS-C6504-E,164.589201,172.267520,164.781000,169.380210,2700,NaN,NaN,NaN,0.061030,0.062733,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
105,swiOT2,8201-24H8FH,110.206985,156.885401,112.000000,158.175901,2000,0.545455,0.645652,26.0,0.056000,0.079088,163.217524,0.081609,0.545455,0.645652,0.645652,252.794818,17.381084,0.064332,0.787826,207.174701,45.620116,0.180463,0.827826,197.164153,55.630665,0.220063,0.857826,190.268899,62.525919,0.247339,0.880398,185.390550,67.404268,0.266636,0.918582,177.684312,75.11